# 01 — Exploratory Data Analysis
**Structural Damage Classification — Gempa Gorkha, Nepal 2015**

Tujuan notebook ini:
1. Memahami struktur data, tipe kolom, dan kualitas data.
2. Melihat distribusi target `damage_grade` (Grade 1–5).
3. Mengamati hubungan karakteristik bangunan dengan tingkat kerusakan.
4. Menguji hipotesis bahwa **lokasi** (proksi intensitas guncangan) adalah sinyal terkuat, termasuk
   kode wilayah lama yang tersembunyi di `building_id`.

Semua gambar disimpan ke `reports/figures/`.

In [ ]:
# === Setup: temukan folder repo (lokal / Kaggle) ===
import sys, subprocess
from pathlib import Path

# Jika dijalankan di Kaggle dan repo belum ada, isi URL GitHub repo kamu (Internet harus ON):
REPO_URL = ""   # contoh: "https://github.com/username/structural-damage-classification.git"

def find_repo_root():
    cands = [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/repo")]
    kin = Path("/kaggle/input")
    if kin.exists():
        cands += [p.parent.parent for p in kin.glob("**/scripts/__init__.py")]
    for c in cands:
        if (c / "scripts" / "__init__.py").exists():
            return c.resolve()
    return None

ROOT = find_repo_root()
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/repo"], check=True)
    ROOT = Path("/kaggle/working/repo")
assert ROOT is not None, "Folder repo (berisi scripts/) tidak ditemukan. Isi REPO_URL atau upload repo sebagai Kaggle Dataset."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold

from scripts.preprocess import load_data, output_dir, parse_grade, TARGET, CAT_COLS, NUM_COLS, LOC_COLS

FIG = output_dir() / "reports" / "figures"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

train, test, sample = load_data()
y = parse_grade(train[TARGET])
print("train:", train.shape, " test:", test.shape, " sample:", sample.shape)
train.head()

## 1. Struktur & kualitas data

In [ ]:
info = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "n_unique": train.nunique(),
    "n_missing": train.isna().sum(),
})
print("duplikat building_id:", train["building_id"].duplicated().sum())
print("building_id test yang juga ada di train:", test["building_id"].isin(train["building_id"]).sum())
info

In [ ]:
train[NUM_COLS].describe().T

## 2. Distribusi target
Kelas tidak seimbang: Grade 5 paling banyak. Karena metriknya **macro F1** (rata-rata F1 tiap kelas
dengan bobot sama), kelas kecil seperti Grade 1–2 sama pentingnya dengan Grade 5.

In [ ]:
dist = pd.Series(y).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(dist.index.astype(str), dist.values, color="#4C72B0")
for i, v in enumerate(dist.values):
    ax.text(i, v, f"{v / dist.sum():.1%}", ha="center", va="bottom")
ax.set_xlabel("damage_grade"); ax.set_ylabel("jumlah bangunan"); ax.set_title("Distribusi target")
fig.tight_layout(); fig.savefig(FIG / "target_distribution.png"); plt.show()

## 3. Fitur numerik vs kerusakan

In [ ]:
df = train.assign(grade=y)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, col in zip(axes, NUM_COLS):
    clip = df[col].clip(upper=df[col].quantile(0.99))
    ax.boxplot([clip[df.grade == g] for g in range(1, 6)], showfliers=False)
    ax.set_xticklabels(range(1, 6)); ax.set_title(col); ax.set_xlabel("grade")
fig.suptitle("Fitur numerik per grade (dipotong di persentil 99)")
fig.tight_layout(); fig.savefig(FIG / "numeric_by_grade.png"); plt.show()

## 4. Fitur kategorikal vs kerusakan
Proporsi grade per kategori untuk material struktur, tipe fondasi, dan tipe atap.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, col in zip(axes, ["superstructure", "foundation_type", "roof_type"]):
    tab = pd.crosstab(df[col], df.grade, normalize="index")
    tab = tab.loc[df.groupby(col).grade.mean().sort_values().index]
    left = np.zeros(len(tab))
    for g in tab.columns:
        ax.barh(tab.index.astype(str).str.replace("has_superstructure_", ""), tab[g], left=left, label=f"G{g}")
        left += tab[g].to_numpy()
    ax.set_title(col); ax.set_xlim(0, 1)
axes[0].legend(ncol=5, loc="lower center", bbox_to_anchor=(1.6, -0.18))
fig.tight_layout(); fig.savefig(FIG / "categorical_by_grade.png", bbox_inches="tight"); plt.show()

## 5. Lokasi: proksi intensitas guncangan
Data tidak memuat rekaman guncangan tanah. Satu-satunya informasi seismik adalah **lokasi**:
kerusakan rata-rata sangat berbeda antar distrik.

In [ ]:
by_d = df.groupby("district_id").grade.agg(["mean", "size"]).sort_values("mean")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(by_d.index.astype(str), by_d["mean"], color="#DD8452")
ax.set_xlabel("district_id"); ax.set_ylabel("rata-rata grade"); ax.set_title("Rata-rata kerusakan per distrik")
fig.tight_layout(); fig.savefig(FIG / "damage_by_district.png"); plt.show()
by_d.T

### 5.1 Kode wilayah lama di `building_id`
`building_id` 12 digit. Prefiksnya **bukan** `ward_id` baru, melainkan kode administrasi lama
(2 digit = distrik, 4 = VDC lama, 6 = ward lama, lebih panjang = blok lebih kecil).
Kita ukur seberapa banyak tiap tingkat menjelaskan target dengan baseline sederhana:
*tebak grade terbanyak di grupnya* (dihitung out-of-fold agar jujur).

In [ ]:
def oof_mode_accuracy(keys, target, n_splits=5, seed=0):
    keys = np.asarray(keys); acc = []
    glob_mode = pd.Series(target).mode()[0]
    for tr, va in KFold(n_splits, shuffle=True, random_state=seed).split(keys):
        tab = pd.crosstab(keys[tr], target[tr])
        mode = pd.Series(tab.columns.to_numpy()[tab.to_numpy().argmax(1)], index=tab.index)
        pred = pd.Series(keys[va]).map(mode).fillna(glob_mode).to_numpy()
        acc.append((pred == target[va]).mean())
    return float(np.mean(acc))

sid = train["building_id"].astype(np.int64).astype(str).str.zfill(12)
rows = [("global mode", 1, (y == pd.Series(y).mode()[0]).mean())]
for col in LOC_COLS:
    rows.append((col, train[col].nunique(), oof_mode_accuracy(train[col], y)))
for k in (2, 4, 6, 8, 9, 10):
    rows.append((f"building_id[:{k}]", sid.str[:k].nunique(), oof_mode_accuracy(sid.str[:k], y)))
gran = pd.DataFrame(rows, columns=["kunci", "jumlah grup", "akurasi OOF (mode grup)"])
gran

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(gran["kunci"][::-1], gran["akurasi OOF (mode grup)"][::-1], color="#55A868")
ax.set_xlabel("akurasi out-of-fold"); ax.set_title("Seberapa informatif tiap level lokasi?")
fig.tight_layout(); fig.savefig(FIG / "location_granularity.png"); plt.show()

### 5.2 Tetangga nomor ID
Nomor survei diberikan berurutan di dalam ward, sehingga bangunan dengan ID berdekatan cenderung
berdekatan secara fisik. Bandingkan kemiripan grade antara pasangan ID bersebelahan vs pasangan acak di
ward lama yang sama.

In [ ]:
d = pd.DataFrame({"bid": train["building_id"].astype(np.int64), "g6": sid.str[:6], "y": y}).sort_values("bid")
same = d.g6.eq(d.g6.shift())
adj = (d.y.eq(d.y.shift()))[same].mean()
rnd = (d.y.to_numpy() == d.groupby("g6").y.transform(lambda s: s.sample(frac=1, random_state=0).to_numpy()).to_numpy()).mean()
print(f"grade sama, ID bersebelahan : {adj:.3f}")
print(f"grade sama, pasangan acak   : {rnd:.3f}")

## 6. Ringkasan temuan
* Target tidak seimbang (Grade 5 ≈ 36%, Grade 1 ≈ 10%) → optimasi khusus macro F1 diperlukan.
* Material struktur adalah faktor konstruksi terkuat: lumpur/batu jauh lebih rusak daripada beton bertulang/semen-bata.
* **Lokasi** adalah sinyal terbesar, karena menggantikan intensitas guncangan yang tidak terekam.
* Prefiks `building_id` memberi level wilayah yang lebih halus dari `ward_id`, dan tetangga ID membawa sinyal tambahan.

Temuan ini menjadi dasar fitur di `scripts/feature_engineering.py`
(target encoding berjenjang, fitur tetangga ID, fitur relatif terhadap area, dan proksi dinamika struktur).